In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 🌟 MODIFICACIÓN AQUÍ: Inicializar Spark cargando explícitamente los paquetes de Delta Lake
spark = SparkSession.builder \
    .appName("IoT-Lakehouse-Gold") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

SILVER_PATH_SPARK = "file:///home/jovyan/work/lakehouse/silver/telemetry_clean"
GOLD_PATH_SPARK = "file:///home/jovyan/work/lakehouse/gold/telemetry_features"

print("🏆 Leyendo datos de la capa Silver para ingeniería de características...")

# Ahora sí encontrará el formato "delta" sin problemas
df_silver = spark.read.format("delta").load(SILVER_PATH_SPARK)

# 2. TRANSFORMACIÓN ANALÍTICA: Crear ventana de tiempo móvil por sensor
# Ordenamos por timestamp para calcular promedios de los últimos 5 registros anteriores más el actual
window_movil = Window.partitionBy("sensor_id").orderBy("timestamp").rowsBetween(-5, 0)

df_gold = df_silver \
    .withColumn("temp_promedio_movil", F.avg("temperatura_limpia").over(window_movil)) \
    .withColumn("vib_desviacion_movil", F.stddev_samp("vibracion_limpia").over(window_movil)) \
    .withColumn("volt_max", F.max("voltaje").over(window_movil)) \
    .withColumn("volt_min", F.min("voltaje").over(window_movil)) \
    .withColumn("volt_delta", F.col("volt_max") - F.col("volt_min"))

# Limpiar nulos iniciales producto del cálculo de la desviación estándar (los primeros registros no tienen historial)
df_gold_clean = df_gold.na.fill(0.0)

# 3. VALIDACIÓN Y GUARDADO
print(f"📊 Features generadas. Total de registros para IA: {df_gold_clean.count()}")

# Guardamos en la capa final GOLD
df_gold_clean.write.format("delta").mode("overwrite").save(GOLD_PATH_SPARK)
print(f"💾 ¡Capa GOLD completada con éxito en Delta Lake! Ruta: {GOLD_PATH_SPARK}")

🏆 Leyendo datos de la capa Silver para ingeniería de características...
📊 Features generadas. Total de registros para IA: 600
💾 ¡Capa GOLD completada con éxito en Delta Lake! Ruta: file:///home/jovyan/work/lakehouse/gold/telemetry_features
